# Module 21: Interactive Metaprogramming, Descriptors & Memory

### What You Will Discover
By running this notebook, you will explore the Descriptor Protocol (`__get__`, `__set__`), inspect the 5-step attribute lookup order, enforce class invariants with `__init_subclass__`, and manipulate binary buffers with zero-copy `memoryview`.

**Key Question Answered:** *Why does a Data Descriptor take precedence over an instance dictionary attribute, while a Non-Data Descriptor is overridden by it?*


In [ ]:
# Step 1: Defining a Typed Data Descriptor
class NonNegative:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get(self.name, 0)

    def __set__(self, instance, value):
        if value < 0:
            raise ValueError(f'{self.name} cannot be negative')
        instance.__dict__[self.name] = value


In [ ]:
# Step 2: Attaching to host class
class InventoryItem:
    price = NonNegative()
    stock = NonNegative()

    def __init__(self, name: str, price: int, stock: int):
        self.name = name
        self.price = price
        self.stock = stock


In [ ]:
# Step 3: Instantiating and testing validation
item = InventoryItem('Laptop', 1200, 15)
print(f'Item: {item.name}, Price: ${item.price}, Stock: {item.stock}')


### 🔮 Prediction Prompt
**Before running the next cell:** If you manually write directly into the instance dictionary `item.__dict__['price'] = -500`, what will `item.price` return on the next line? Explain why in terms of descriptor priority.


In [ ]:
# Surprising Result: Data Descriptors Override Instance Dict Lookups
item.__dict__['price'] = 9999
print(f'Value via item.price: {item.price}')
print('Explanation: Because NonNegative implements __set__, it is a DATA DESCRIPTOR.')
print('Python checks Data Descriptors on the class MRO BEFORE checking instance __dict__!')


### Class Customization via `__init_subclass__` (PEP 487)
Modern class registration without complicated metaclass MRO conflicts.


In [ ]:
class PluginRegistry:
    plugins = {}  # noqa: RUF012
    def __init_subclass__(cls, plugin_name: str, **kwargs):
        super().__init_subclass__(**kwargs)
        cls.plugins[plugin_name] = cls

class JSONPlugin(PluginRegistry, plugin_name='json'):
    pass
class TOMLPlugin(PluginRegistry, plugin_name='toml'):
    pass
print(f'Registered plugins: {list(PluginRegistry.plugins.keys())}')


### Zero-Copy Slicing with `memoryview`
Avoid copying large binary arrays when slicing buffers.


In [ ]:
import sys

raw_bytes = bytearray(b'HEADER' + b'X' * 1000 + b'FOOTER')
mv = memoryview(raw_bytes)
slice_view = mv[6:1006]
print(f'Original size: {sys.getsizeof(raw_bytes)} bytes')
print(f'memoryview slice object size: {sys.getsizeof(slice_view)} bytes (Zero byte copying!)')


### 🛠️ Interactive Challenge: Fix the Descriptor Leak
The following descriptor stores state on `self.value` instead of `instance.__dict__`, meaning all instances of the class share the exact same value. Fix it to store state in `instance.__dict__`.


In [ ]:
# TODO: FIX ME - Store state in instance.__dict__ to prevent instance sharing
class BrokenField:
    def __set_name__(self, owner, name):
        self.name = name
    def __get__(self, instance, owner):
        if instance is None:
            return self
        # FIX: return instance.__dict__.get(self.name)
        return instance.__dict__.get(self.name)
    def __set__(self, instance, value):
        # FIX: instance.__dict__[self.name] = value
        instance.__dict__[self.name] = value

class Customer:
    age = BrokenField()

c1 = Customer()
c1.age = 20
c2 = Customer()
c2.age = 45
print(f'c1 age: {c1.age}, c2 age: {c2.age} (Distinct instances!)')


### 🏁 Summary & Next Steps
- Data descriptors implement `__set__` and override instance dictionaries.
- Non-data descriptors implement only `__get__` (e.g. methods).
- Use `__init_subclass__` instead of metaclasses for class hooks.
- Run `python 01_descriptors_protocol_demo.py` and `02_metaclasses_and_init_subclass_demo.py`.
- Follow [PROJECT_GUIDE.md](PROJECT_GUIDE.md) to implement the zero-copy model DSL.
